[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/02-data-harmonization/03-building_a_custom_character_mapping.ipynb)

# Building a Custom Character Mapping

`create_english_standard()` and `create_german_standard()` cover the most common cases, but real datasets often need something more specific, part numbers with OCR errors, identifiers that shouldn't be case-sensitive in one direction but should preserve certain symbols, or leetspeak-style digit substitutions in scanned documents.

`CharacterMapping` isn't limited to its factory methods. You can construct one directly, with full control over every normalization rule.

In this notebook you will:

1. Review what each `CharacterMapping` parameter actually controls
2. Build a custom mapping for a real problem the built-in standards don't solve: leetspeak-style OCR errors in part numbers
3. Attach it to an index and confirm it works
4. Save your custom mapping to JSON so it can be reused across a pipeline


## Install the SDK

M|BOX is distributed on PyPI. Run the cell below to install it (or run this in your terminal without the `!`).

In [ ]:
# !pip install mbox

## 1. The building blocks

Every `CharacterMapping` is built from the same set of parameters. The factory methods just set sensible defaults for common languages, you can set every one of these yourself:

| Parameter | Default | What it does |
|---|---|---|
| `name` | `"Default Mapping"` | A descriptive identifier for the mapping |
| `mapped_characters` | `"A-Za-z 0-9"` | Which characters are valid tokens; anything outside this set is normalized away |
| `map_upper` | `True` | Converts lowercase to uppercase |
| `deaccentuate` | `True` | Strips diacritics (`é` → `e`, `ñ` → `n`) |
| `expand_umlauts` | `False` | Expands Germanic umlauts and ligatures (`ä` → `AE`) instead of just stripping them |
| `map_currency` | `True` | Normalizes currency symbols to a uniform representation |
| `map_spaces` | `True` | Collapses multiple whitespace, tabs, and non-breaking spaces into a single space |
| `any_to_latin` | `False` | Transliterates non-Latin scripts to Latin equivalents |
| `numbers_as_characters` | `False` | Converts leetspeak digits to their letter equivalents (`3` → `E`, `0` → `O`, `7` → `T`) |

You mix and match these to describe exactly how a specific field should be normalized.

## 2. A problem the standard mappings don't solve

Suppose you're matching part numbers extracted from scanned invoices via OCR. OCR software regularly confuses visually similar characters, `O` and `0`, `B` and `8`, `E` and `3`, and sometimes those substitutions look exactly like intentional leetspeak.

Let's index some clean part numbers, then search using an OCR-garbled version of one.

In [1]:
import os
import pandas as pd
from mbox.indexing import TableIndexer

df = pd.DataFrame({
    "part_number": ["B88-EXT", "A12-PWR", "C99-SNS", "D45-REL"],
    "description": ["High power battery pack", "Solar panel controller", "Proximity sensor unit", "12V relay switch"]
})

baseline_index = TableIndexer.create_index(df, index_columns=["part_number"], tmp_dir="tmp_index")

# OCR misread "B88-EXT" as "8B8-3XT" -- B/8 and E/3 confusion
baseline_results = baseline_index.match(
    part_number="8B8-3XT",
    include_field_scores=True
)

baseline_results

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


,query_row,index_row,part_number_candidate,description_candidate,overall_score,part_number_score
0,0,0,B88-EXT,High power battery pack,36,36


`part_number_score` should come out low. Three characters differ (`B`→`8`, `8`→`B`, `E`→`3`), and none of the built-in factory methods know that `8` and `B`, or `3` and `E`, are meant to be treated as equivalent in this context. Neither `create_english_standard()` nor `create_german_standard()` was designed with OCR-style digit substitution in mind. This needs a mapping built specifically for the problem.

## 3. Building a mapping for leetspeak / OCR substitution

`numbers_as_characters=True` is exactly the setting for this: it converts common leetspeak digits into their character equivalents, so `3` and `E` (for example) both normalize to the same token. Combined with restricting `mapped_characters` to just what part numbers actually use, this gives us a purpose-built mapping.

In [2]:
from mbox.mapping import CharacterMapping

part_number_mapping = CharacterMapping(
    name="part_number_ocr_cleaner",
    mapped_characters="A-Z0-9-",
    map_upper=True,
    deaccentuate=True,
    numbers_as_characters=True
)

print("B88-EXT ->", part_number_mapping.apply("B88-EXT"))
print("8B8-3XT ->", part_number_mapping.apply("8B8-3XT"))

B88-EXT -> BBB-EXT
8B8-3XT -> BBB-EXT


If both lines print the same normalized string, the mapping is doing its job: the OCR-garbled version and the correctly recognized version now collapse to an identical representation, the same way `Müller` and `Mueller` did in the previous notebook.

## 4. Attach it to the index and re-run the query

In [3]:
harmonized_index = TableIndexer.create_index(
    df=df,
    index_columns=["part_number"],
    character_mappings={"part_number": part_number_mapping},
    tmp_dir="tmp_index"
)

harmonized_results = harmonized_index.match(
    part_number="8B8-3XT",
    include_field_scores=True
)

harmonized_results

,query_row,index_row,part_number_candidate,description_candidate,overall_score,part_number_score
0,0,0,B88-EXT,High power battery pack,100,100


Compare `part_number_score` against the baseline in Step 2. The OCR-confused query should now resolve to `"B88-EXT"` with a meaningfully higher score, because the mapping treats the digit/letter substitutions it was designed for as free, rather than as real character edits.

## 5. `mapped_characters` as a strict filter

There's a second, quieter thing happening in the mapping above: `mapped_characters="A-Z0-9-"` explicitly includes the hyphen. If you left it at the default (`"A-Za-z 0-9"`), the hyphen would be stripped out entirely during normalization, and `"B88-EXT"` would become `"B88EXT"`.

This matters whenever a field has structural characters that carry meaning, hyphens in part numbers, slashes in dates, `@` in identifiers. `mapped_characters` is where you declare which of those matter enough to keep.

In [4]:
default_mapped = CharacterMapping(name="default_charset")
custom_mapped = CharacterMapping(name="keeps_hyphen", mapped_characters="A-Z0-9-")

print("Default mapped_characters ->", default_mapped.apply("B88-EXT"))
print("Custom mapped_characters  ->", custom_mapped.apply("B88-EXT"))

Default mapped_characters -> B88 EXT
Custom mapped_characters  -> B88-EXT


## 6. Saving your custom mapping for reuse

Once you've tuned a mapping for a specific field, save it to JSON so it can be version-controlled and loaded consistently across notebooks, services, or pipeline runs, rather than redefining the same parameters by hand every time.

In [5]:
os.makedirs("mappings", exist_ok=True)
part_number_mapping.to_json("./mappings/part_number_ocr_cleaner.json")

# Reload it anywhere else in your pipeline
loaded_mapping = CharacterMapping.from_json("./mappings/part_number_ocr_cleaner.json")
loaded_mapping.to_dict()

{'name': 'part_number_ocr_cleaner',
 'mapped_characters': 'A-Z0-9-',
 'map_upper': True,
 'deaccentuate': True,
 'expand_umlauts': False,
 'map_currency': True,
 'map_spaces': True,
 'any_to_latin': False,
 'numbers_as_characters': True}

## 7. When to build custom vs. use a factory method

| If your field... | Use |
|---|---|
| Is English text with occasional accents | `CharacterMapping.create_english_standard()` |
| Contains German umlauts or `ß` | `CharacterMapping.create_german_standard()` |
| Should ignore everything except letters and digits | `CharacterMapping.create_strict_alphanumeric()` |
| Has structural characters that must be preserved (hyphens, slashes) | Custom, with `mapped_characters` set explicitly |
| Comes from OCR, scanned documents, or leetspeak-prone sources | Custom, with `numbers_as_characters=True` |
| Mixes multiple scripts (e.g. Cyrillic, Han characters) that need Latin comparison | Custom, with `any_to_latin=True` |

A good default approach: start with the closest factory method, apply it to a sample of your real data with `.apply_series()`, inspect the output, and only build a custom mapping once you find cases the factory doesn't handle.

## Next steps

- **`combining_mappings_and_aliases_in_one_index.ipynb`**, use a custom `CharacterMapping` alongside an `AliasSet` on the same field
- **`03-index-configuration/`**, attach custom mappings explicitly via `TableFieldConfig` as part of a versioned schema

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*